# ADPO evaluation on real GitHub Actions data

Reproducible evaluation of ADPO's six deterministic analyzers on CI history from the
public repositories in `repos.yaml`. Every number in `REPORT.md` is computed in this
notebook (the last cell writes it). Regenerate with the command in `research/README.md`.

In [1]:
import json
import math
import sys
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt
from scipy import stats

HERE = Path.cwd() if (Path.cwd() / "repos.yaml").exists() else Path.cwd() / "research"
sys.path.insert(0, str(HERE))
from adpo_data import load_tables, to_raw_runs  # noqa: E402

from adpo.analyzers.dependency_install import DependencyInstallAnalyzer, DependencyInstallConfig
from adpo.analyzers.failure_retry import FailureRetryAnalyzer, FailureRetryConfig
from adpo.analyzers.historical_regression import HistoricalRegressionAnalyzer
from adpo.analyzers.parallelization import ParallelizationAnalyzer, ParallelizationConfig
from adpo.analyzers.slow_job import SlowJobAnalyzer, SlowJobConfig
from adpo.analyzers.slow_step import SlowStepAnalyzer, SlowStepConfig
from adpo.engine import AnalysisEngine
from adpo.ingest import parse_workflow_runs

warnings.filterwarnings("ignore", category=RuntimeWarning)
FIG = HERE / "figures"
FIG.mkdir(exist_ok=True)
SEED = 20261112
R = {}  # every number quoted in REPORT.md is stored here
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png")
    plt.close()


tables, SOURCE = load_tables()
meta = json.loads((HERE / "data" / "collection_meta.json").read_text())
REPOS = sorted(tables["runs"].repo.unique())
RUNS = {repo: parse_workflow_runs(to_raw_runs(tables, repo)) for repo in REPOS}
R["source"] = SOURCE
R["collected_at"] = meta["collected_at"]
R["n_repos"] = len(REPOS)
print(SOURCE, len(REPOS), "repos")

raw 1 repos


## a. Data description

In [2]:
runs_df = tables["runs"].copy()
jobs_df = tables["jobs"].copy()
steps_df = tables["steps"].copy()
runs_df["base_run"] = runs_df.run_id.str.split("/").str[0]
for df in (jobs_df, steps_df):
    df["started"] = pd.to_datetime(df.started_at, utc=True, errors="coerce")
    df["completed"] = pd.to_datetime(df.completed_at, utc=True, errors="coerce")
    df["duration"] = (df.completed - df.started).dt.total_seconds()
jobs_df = jobs_df.merge(runs_df[["repo", "run_id", "base_run", "created_at", "event", "branch"]], on=["repo", "run_id"])

per_repo = pd.DataFrame(
    {
        "workflow": [meta["repos"][r]["workflow"] for r in REPOS],
        "runs": runs_df.groupby("repo").base_run.nunique(),
        "attempts": runs_df.groupby("repo").size(),
        "jobs": jobs_df.groupby("repo").size(),
        "steps": steps_df.groupby("repo").size(),
        "first_run": runs_df.groupby("repo").created_at.min().str[:10],
        "last_run": runs_df.groupby("repo").created_at.max().str[:10],
        "jobs_matched_to_yaml_pct": (jobs_df.groupby("repo").yaml_key.apply(lambda s: s.notna().mean()) * 100).round(0),
    },
    index=REPOS,
)
R["totals"] = {k: int(per_repo[k].sum()) for k in ("runs", "attempts", "jobs", "steps")}
R["per_repo_table"] = per_repo.to_markdown()
R["yaml_match_pct_overall"] = round(float(jobs_df.yaml_key.notna().mean() * 100), 1)

conc = jobs_df.conclusion.fillna("none").value_counts(normalize=True).mul(100).round(1)
R["job_conclusion_pct"] = conc.to_dict()
run_conc = runs_df[~runs_df.run_id.str.contains("/")].conclusion.fillna("none").value_counts(normalize=True)
R["run_conclusion_pct"] = run_conc.mul(100).round(1).to_dict()
R["event_pct"] = runs_df[~runs_df.run_id.str.contains("/")].event.value_counts(normalize=True).mul(100).round(1).to_dict()

missing_job_timing = jobs_df.duration.isna().mean() * 100
negative = (jobs_df.duration < 0).sum()
step_missing = steps_df.duration.isna().mean() * 100
R["missing"] = {
    "job_timing_missing_pct": round(float(missing_job_timing), 2),
    "job_negative_durations": int(negative),
    "step_timing_missing_pct": round(float(step_missing), 2),
    "skipped_jobs_with_zero_duration": int(((jobs_df.conclusion == "skipped") & (jobs_df.duration.fillna(0) <= 1)).sum()),
}
R["retried_runs"] = int((runs_df.run_attempt > 1).sum())
print(per_repo)
print(R["job_conclusion_pct"], R["missing"], R["event_pct"])

                 workflow  runs  attempts  jobs  steps   first_run  \
pallets/flask  tests.yaml   150       150  1855  10983  2026-03-02   

                 last_run  jobs_matched_to_yaml_pct  
pallets/flask  2026-10-30                     100.0  
{'success': 79.5, 'failure': 19.4, 'cancelled': 1.1} {'job_timing_missing_pct': 0.0, 'job_negative_durations': 0, 'step_timing_missing_pct': 0.0, 'skipped_jobs_with_zero_duration': 0} {'pull_request': 77.3, 'push': 22.7}


## b. Distribution analysis

Series = successful executions of one job (repo x job display name), chronological.

In [3]:
succ = jobs_df[(jobs_df.conclusion == "success") & jobs_df.duration.notna() & (jobs_df.duration > 0)]
series = {k: g.sort_values("created_at").duration.to_numpy() for k, g in succ.groupby(["repo", "name"]) if len(g) >= 20}
R["n_series_20"] = len(series)

skews = np.array([stats.skew(v) for v in series.values()])
log_skews = np.array([stats.skew(np.log(v)) for v in series.values()])
mean_over_median = np.array([v.mean() / np.median(v) for v in series.values()])
R["skew"] = {
    "median_skew": round(float(np.median(skews)), 2),
    "pct_skew_gt_1": round(float((skews > 1).mean() * 100), 1),
    "median_log_skew": round(float(np.median(log_skews)), 2),
    "pct_mean_gt_median_10pct": round(float((mean_over_median > 1.10).mean() * 100), 1),
    "pct_shapiro_reject_normal": round(
        float(np.mean([stats.shapiro(v[:500]).pvalue < 0.05 for v in series.values()]) * 100), 1
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
pooled = np.concatenate([v / np.median(v) for v in series.values()])
axes[0].hist(np.clip(pooled, 0, 4), bins=80, color="#4C72B0")
axes[0].axvline(1, color="k", lw=0.8)
axes[0].set(title="Job duration / its own median (pooled)", xlabel="ratio (clipped at 4)", ylabel="job executions")
axes[1].hist(np.clip(skews, -2, 10), bins=40, color="#4C72B0", alpha=0.8, label="raw seconds")
axes[1].hist(np.clip(log_skews, -2, 10), bins=40, color="#DD8452", alpha=0.6, label="log seconds")
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set(title="Per-series sample skewness", xlabel="skewness (clipped)", ylabel="series")
axes[1].legend(frameon=False)
savefig("b1_skew")
print(R["skew"])

{'median_skew': 0.58, 'pct_skew_gt_1': 38.5, 'median_log_skew': 0.19, 'pct_mean_gt_median_10pct': 0.0, 'pct_shapiro_reject_normal': 76.9}


### Threshold estimators: mean+2·SD vs median+2·(1.4826·MAD) vs empirical 97.7th percentile

All three target the same one-sided tail (2.3% under normality). Fit on the first 2/3 of each
series, measure the exceedance rate on the last 1/3 (calibration) and the bootstrap spread of
the threshold relative to the median (stability).

In [4]:
def thr_mean(x):
    return x.mean() + 2 * x.std(ddof=1)


def thr_mad(x):
    return np.median(x) + 2 * 1.4826 * stats.median_abs_deviation(x)


def thr_pct(x):
    return np.percentile(x, 97.7)


ESTIMATORS = {"mean+2sd": thr_mean, "median+2MAD": thr_mad, "p97.7": thr_pct}
rng = np.random.default_rng(SEED)
rows = []
for key, v in series.items():
    if len(v) < 30:
        continue
    cut = len(v) * 2 // 3
    train, test = v[:cut], v[cut:]
    for name, f in ESTIMATORS.items():
        t = f(train)
        boots = [f(rng.choice(train, size=len(train), replace=True)) for _ in range(200)]
        rows.append(
            {
                "series": key,
                "estimator": name,
                "exceed_test": float((test > t).mean()),
                "rel_boot_sd": float(np.std(boots) / np.median(train)),
                "thr_over_median": float(t / np.median(train)),
            }
        )
thr = pd.DataFrame(rows)
thr_summary = thr.groupby("estimator").agg(
    series=("series", "nunique"),
    median_exceed_pct=("exceed_test", lambda s: round(s.median() * 100, 1)),
    mean_exceed_pct=("exceed_test", lambda s: round(s.mean() * 100, 1)),
    pct_series_exceed_over_10pct=("exceed_test", lambda s: round((s > 0.10).mean() * 100, 1)),
    median_rel_boot_sd=("rel_boot_sd", lambda s: round(s.median(), 3)),
    median_thr_over_median=("thr_over_median", lambda s: round(s.median(), 2)),
)
R["thr_table"] = thr_summary.to_markdown()
R["thr"] = thr_summary.to_dict(orient="index")

fig, ax = plt.subplots(figsize=(6, 3.2))
for name in ESTIMATORS:
    s = np.sort(thr[thr.estimator == name].exceed_test.to_numpy())
    ax.plot(s, np.linspace(0, 1, len(s)), label=name)
ax.axvline(0.023, color="k", ls="--", lw=0.8, label="nominal 2.3%")
ax.set(xlim=(0, 0.4), xlabel="exceedance rate on held-out runs", ylabel="cumulative share of series",
       title="Out-of-sample calibration of slow-run thresholds")
ax.legend(frameon=False)
savefig("b2_threshold_calibration")
print(thr_summary)

             series  median_exceed_pct  mean_exceed_pct  \
estimator                                                 
mean+2sd         13                4.7              4.1   
median+2MAD      13                7.1              8.2   
p97.7            13                4.7              3.9   

             pct_series_exceed_over_10pct  median_rel_boot_sd  \
estimator                                                       
mean+2sd                              0.0               0.041   
median+2MAD                          30.8               0.077   
p97.7                                 0.0               0.062   

             median_thr_over_median  
estimator                            
mean+2sd                       1.37  
median+2MAD                    1.37  
p97.7                          1.36  


### Coefficient of variation (SD/mean) vs a robust analogue (1.4826·MAD/median)

Several analyzers branch on CV: savings are `unknown` when CV < 0.10, the "high variance"
recommendation fires at CV >= 0.25, and Dependency Install calls a cached step
"possibly ineffective" at CV >= 0.30. One slow outlier can push SD/mean over these lines.

In [5]:
def robust_cv(x):
    m = np.median(x)
    return 1.4826 * stats.median_abs_deviation(x) / m if m > 0 else np.nan


cv_rows = [{"series": k, "cv": np.std(v, ddof=1) / np.mean(v), "rcv": robust_cv(v)} for k, v in series.items()]
cvdf = pd.DataFrame(cv_rows)
R["cv"] = {}
for t in (0.10, 0.25, 0.30):
    a, b = cvdf.cv >= t, cvdf.rcv >= t
    R["cv"][str(t)] = {
        "cv_ge": round(float(a.mean() * 100), 1),
        "robust_ge": round(float(b.mean() * 100), 1),
        "disagree": round(float((a != b).mean() * 100), 1),
        "cv_only": int((a & ~b).sum()),
        "robust_only": int((~a & b).sum()),
    }
# Outlier sensitivity: add one run at 3x the median to every series.
bumped = [
    (np.std(np.append(v, 3 * np.median(v)), ddof=1) / np.mean(np.append(v, 3 * np.median(v))) - np.std(v, ddof=1) / np.mean(v),
     robust_cv(np.append(v, 3 * np.median(v))) - robust_cv(v))
    for v in series.values()
]
bumped = np.array(bumped)
R["cv_one_outlier_median_shift"] = {"cv": round(float(np.median(bumped[:, 0])), 3), "robust": round(float(np.median(bumped[:, 1])), 3)}

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.scatter(cvdf.rcv.clip(0, 1.2), cvdf.cv.clip(0, 1.2), s=6, alpha=0.5)
ax.plot([0, 1.2], [0, 1.2], color="k", lw=0.7)
for t in (0.10, 0.30):
    ax.axhline(t, color="grey", ls=":", lw=0.7)
    ax.axvline(t, color="grey", ls=":", lw=0.7)
ax.set(xlabel="robust CV (1.4826·MAD/median)", ylabel="CV (SD/mean)", title="CV vs robust CV per job series")
savefig("b3_cv_vs_robust")
print(R["cv"], R["cv_one_outlier_median_shift"])

{'0.1': {'cv_ge': 100.0, 'robust_ge': 100.0, 'disagree': 0.0, 'cv_only': 0, 'robust_only': 0}, '0.25': {'cv_ge': 23.1, 'robust_ge': 7.7, 'disagree': 30.8, 'cv_only': 3, 'robust_only': 1}, '0.3': {'cv_ge': 15.4, 'robust_ge': 0.0, 'disagree': 15.4, 'cv_only': 2, 'robust_only': 0}} {'cv': 0.061, 'robust': 0.0}


## c. Sensitivity analysis

Each analyzer is run once per repo with its gates opened as far as its metrics allow; the
threshold grid is then applied to the metrics each finding reports. Defaults are marked.

In [6]:
def run_all(analyzer):
    out = []
    for repo in REPOS:
        for f in analyzer.analyze(RUNS[repo]):
            out.append({"repo": repo, **f.metrics, "_finding": f})
    return pd.DataFrame(out)


loose_job = run_all(SlowJobAnalyzer(SlowJobConfig(min_duration_seconds=0, absolute_severe_seconds=1e12, min_share_of_workflow=0)))
loose_step = run_all(SlowStepAnalyzer(SlowStepConfig(min_duration_seconds=0, absolute_severe_seconds=1e12, min_share_of_job=0)))
from adpo.analyzers.historical_regression import RegressionConfig  # noqa: E402

loose_reg = run_all(HistoricalRegressionAnalyzer(RegressionConfig(min_pct_increase=-1e9, min_absolute_increase_seconds=-1e9)))
loose_dep = run_all(DependencyInstallAnalyzer(DependencyInstallConfig(min_duration_seconds=0, ineffective_cache_cv_threshold=0)))
loose_fail = run_all(FailureRetryAnalyzer(FailureRetryConfig(flaky_min_rate=0)))
loose_par = run_all(ParallelizationAnalyzer(ParallelizationConfig(min_job_duration_seconds=0, min_sequential_consistency=0)))

d_job, d_step = SlowJobConfig(), SlowStepConfig()


def count_job(share, n_min=1):
    if loose_job.empty:
        return 0
    m, s = loose_job.median_duration_seconds, loose_job.share_of_workflow_median
    hit = ((m >= d_job.min_duration_seconds) & (s >= share)) | (m >= d_job.absolute_severe_seconds)
    return int((hit & (loose_job.sample_size >= n_min)).sum())


def count_step(share, n_min=1):
    if loose_step.empty:
        return 0
    m, s = loose_step.median_duration_seconds, loose_step.share_of_job_median.fillna(-1)
    hit = ((m >= d_step.min_duration_seconds) & (s >= share)) | (m >= d_step.absolute_severe_seconds)
    return int((hit & (loose_step.sample_size >= n_min)).sum())


def count_reg(pct):
    if loose_reg.empty:
        return 0
    return int(((loose_reg.pct_increase >= pct) & (loose_reg.absolute_increase_seconds >= 20)).sum())


def count_dep(cv_t):
    if loose_dep.empty:
        return 0
    cached = loose_dep.cache_action_detected.astype(bool)
    keep = (loose_dep.median_duration_seconds >= 20) & (~cached | (loose_dep.coefficient_of_variation.fillna(0) >= cv_t))
    return int(keep.sum())


flaky = loose_fail[loose_fail["classification"].isin(["flaky", "chronic_failure"])] if len(loose_fail) else loose_fail


def count_flaky(rate):
    if flaky.empty:
        return 0
    return int((flaky.failure_rate >= rate).sum())


def count_par(cons):
    if loose_par.empty:
        return 0
    ok = (loose_par.job_a_median_seconds >= 20) & (loose_par.job_b_median_seconds >= 20)
    return int((ok & (loose_par.sequential_consistency >= cons)).sum())


grids = {
    "slow_job: min_share_of_workflow": (np.round(np.arange(0.05, 0.61, 0.05), 2), count_job, 0.25),
    "slow_step: min_share_of_job": (np.round(np.arange(0.1, 0.91, 0.1), 2), count_step, 0.4),
    "historical_regression: min_pct_increase": (np.round(np.arange(0.05, 0.81, 0.05), 2), count_reg, 0.25),
    "dependency_install: cache CV threshold": (np.round(np.arange(0.1, 0.81, 0.1), 2), count_dep, 0.30),
    "failure_retry: flaky_min_rate": (np.round([0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3], 2), count_flaky, 0.05),
    "parallelization: min_sequential_consistency": (np.round(np.arange(0.5, 1.01, 0.05), 2), count_par, 0.75),
}
fig, axes = plt.subplots(2, 3, figsize=(10, 5.4))
R["sensitivity"] = {}
for ax, (title, (grid, fn, default)) in zip(axes.flat, grids.items()):
    ys = [fn(g) for g in grid]
    R["sensitivity"][title] = dict(zip([float(g) for g in grid], ys))
    ax.plot(grid, ys, marker="o", ms=3)
    ax.axvline(default, color="crimson", ls="--", lw=0.8)
    ax.set(title=title, xlabel="threshold", ylabel="findings (all repos)")
savefig("c1_sensitivity")

# Small-sample findings at defaults.
R["small_n"] = {
    "slow_job_defaults": count_job(0.25),
    "slow_job_n_lt_5": count_job(0.25) - count_job(0.25, n_min=5),
    "slow_step_defaults": count_step(0.4),
    "slow_step_n_lt_5": count_step(0.4) - count_step(0.4, n_min=5),
}
# Parallelization: how much of the signal depends on the YAML `needs` edges.
par_no_yaml = []
for repo in REPOS:
    stripped = to_raw_runs(tables, repo)
    for r in stripped:
        for j in r["jobs"]:
            j["needs"] = []
    par_no_yaml += ParallelizationAnalyzer().analyze(parse_workflow_runs(stripped))
R["par_with_yaml"] = count_par(0.75)
R["par_without_yaml"] = len(par_no_yaml)
print(json.dumps(R["sensitivity"], indent=1)[:1500], R["small_n"], R["par_with_yaml"], R["par_without_yaml"])

{
 "slow_job: min_share_of_workflow": {
  "0.05": 0,
  "0.1": 0,
  "0.15": 0,
  "0.2": 0,
  "0.25": 0,
  "0.3": 0,
  "0.35": 0,
  "0.4": 0,
  "0.45": 0,
  "0.5": 0,
  "0.55": 0,
  "0.6": 0
 },
 "slow_step: min_share_of_job": {
  "0.1": 0,
  "0.2": 0,
  "0.3": 0,
  "0.4": 0,
  "0.5": 0,
  "0.6": 0,
  "0.7": 0,
  "0.8": 0,
  "0.9": 0
 },
 "historical_regression: min_pct_increase": {
  "0.05": 0,
  "0.1": 0,
  "0.15": 0,
  "0.2": 0,
  "0.25": 0,
  "0.3": 0,
  "0.35": 0,
  "0.4": 0,
  "0.45": 0,
  "0.5": 0,
  "0.55": 0,
  "0.6": 0,
  "0.65": 0,
  "0.7": 0,
  "0.75": 0,
  "0.8": 0
 },
 "dependency_install: cache CV threshold": {
  "0.1": 0,
  "0.2": 0,
  "0.3": 0,
  "0.4": 0,
  "0.5": 0,
  "0.6": 0,
  "0.7": 0,
  "0.8": 0
 },
 "failure_retry: flaky_min_rate": {
  "0.01": 14,
  "0.02": 14,
  "0.05": 13,
  "0.1": 12,
  "0.15": 3,
  "0.2": 2,
  "0.3": 2
 },
 "parallelization: min_sequential_consistency": {
  "0.5": 0,
  "0.55": 0,
  "0.6": 0,
  "0.65": 0,
  "0.7": 0,
  "0.75": 0,
  "0.8": 0,
 

## d. Regression detection: ADPO window rule vs Mann-Whitney U vs change-point

Same series as (b) with n >= 12 successful runs. Methods:
* **ADPO**: recent 40% median vs baseline median, flagged if +25% and +20 s.
* **MWU**: one-sided Mann-Whitney U (recent > baseline) on the same windows, Benjamini-Hochberg q < 0.05.
* **MWU+effect**: MWU significant *and* the ADPO effect-size gates.
* **Change-point**: PELT (L1 cost, log durations, penalty 2·log(n)·σ̂ (BIC-style for an L1/Laplace cost) with σ̂ from the MAD of
  successive differences); flagged if the last segment's median is >= 25% and >= 20 s above the
  previous segment's, and the last segment has >= 3 runs.

In [7]:
reg_an = HistoricalRegressionAnalyzer()
cfg = reg_an.config


def windows(v):
    n = len(v)
    recent_n = min(max(cfg.min_recent_samples, round(n * cfg.recent_fraction)), n - cfg.min_baseline_samples)
    return v[: n - recent_n], v[n - recent_n :]


def changepoint(v):
    x = np.log(v)
    sigma = 1.4826 * stats.median_abs_deviation(np.diff(x)) / math.sqrt(2) or 1e-6
    bkps = rpt.Pelt(model="l1", min_size=3, jump=1).fit(x).predict(pen=2 * math.log(len(x)) * sigma)
    if len(bkps) < 2:
        return False, bkps
    prev = v[(bkps[-3] if len(bkps) >= 3 else 0) : bkps[-2]]
    last = v[bkps[-2] :]
    up = np.median(last) - np.median(prev)
    return bool(len(last) >= 3 and up >= 20 and up / np.median(prev) >= 0.25), bkps


reg_series = {k: v for k, v in
              ((k, g.sort_values("created_at").duration.to_numpy()) for k, g in succ.groupby(["repo", "name"]))
              if len(v) >= 12}
rows = []
for key, v in reg_series.items():
    base, rec = windows(v)
    adpo = reg_an._analyze_series(str(key), [(float(d), str(i)) for i, d in enumerate(v)]) is not None
    p = stats.mannwhitneyu(rec, base, alternative="greater").pvalue
    eff = (np.median(rec) - np.median(base) >= 20) and (np.median(rec) / np.median(base) - 1 >= 0.25)
    cp, bkps = changepoint(v)
    rows.append({"series": key, "n": len(v), "adpo": adpo, "p": p, "effect": eff, "cp": cp, "bkps": bkps,
                 "pct": np.median(rec) / np.median(base) - 1})
reg = pd.DataFrame(rows)
reg["q"] = stats.false_discovery_control(reg.p.to_numpy(), method="bh")
reg["mwu"] = reg.q < 0.05
reg["mwu_eff"] = reg.mwu & reg.effect


def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return round(float((po - pe) / (1 - pe)), 3) if pe < 1 else float("nan")


methods = ["adpo", "mwu", "mwu_eff", "cp"]
R["reg_n_series"] = len(reg)
R["reg_flags"] = {m: int(reg[m].sum()) for m in methods}
R["reg_kappa"] = {f"{a} vs {b}": kappa(reg[a], reg[b]) for i, a in enumerate(methods) for b in methods[i + 1 :]}
R["reg_confusion_adpo_mwu"] = pd.crosstab(reg.adpo, reg.mwu, rownames=["ADPO"], colnames=["MWU q<0.05"]).to_markdown()
R["reg_adpo_not_mwu"] = int((reg.adpo & ~reg.mwu).sum())
R["reg_mwu_eff_not_adpo"] = int((reg.mwu_eff & ~reg.adpo).sum())
R["reg_adpo_not_cp"] = int((reg.adpo & ~reg.cp).sum())
R["reg_cp_not_adpo"] = int((reg.cp & ~reg.adpo).sum())


def pick(mask, k=1):
    sub = reg[mask].sort_values("n", ascending=False)
    return sub.head(k).to_dict("records")


examples = {
    "ADPO flags, MWU does not": pick(reg.adpo & ~reg.mwu),
    "MWU+effect flags, ADPO does not": pick(reg.mwu_eff & ~reg.adpo),
    "Change-point flags, ADPO does not": pick(reg.cp & ~reg.adpo),
    "All methods agree (flag)": pick(reg.adpo & reg.mwu & reg.cp),
}
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6))
R["reg_examples"] = []
for ax, (label, ex) in zip(axes.flat, examples.items()):
    ax.set_title(label, fontsize=9)
    if not ex:
        ax.text(0.5, 0.5, "no such series", ha="center", transform=ax.transAxes)
        continue
    e = ex[0]
    v = reg_series[e["series"]]
    base, rec = windows(v)
    ax.plot(v, ".", ms=3, color="#4C72B0")
    ax.hlines(np.median(base), 0, len(base) - 1, color="k", lw=1)
    ax.hlines(np.median(rec), len(base), len(v) - 1, color="crimson", lw=1)
    for b in e["bkps"][:-1]:
        ax.axvline(b, color="orange", ls="--", lw=0.8)
    ax.set(xlabel="run (chronological)", ylabel="seconds")
    ax.text(0.01, 0.95, f"{e['series'][0]}\n{e['series'][1][:45]}\nq={e['q']:.3f}, Δmedian={e['pct']*100:+.0f}%",
            transform=ax.transAxes, va="top", fontsize=7)
    R["reg_examples"].append({"case": label, "repo": e["series"][0], "job": e["series"][1], "n": int(e["n"]),
                              "q": round(float(e["q"]), 4), "pct": round(float(e["pct"]) * 100, 1),
                              "adpo": bool(e["adpo"]), "mwu": bool(e["mwu"]), "cp": bool(e["cp"])})
savefig("d1_regression_examples")
print(R["reg_flags"], R["reg_kappa"])
print(R["reg_confusion_adpo_mwu"])

{'adpo': 0, 'mwu': 1, 'mwu_eff': 0, 'cp': 0} {'adpo vs mwu': 0.0, 'adpo vs mwu_eff': nan, 'adpo vs cp': nan, 'mwu vs mwu_eff': 0.0, 'mwu vs cp': 0.0, 'mwu_eff vs cp': nan}
| ADPO   |   False |   True |
|:-------|--------:|-------:|
| False  |      13 |      1 |


## e. Flakiness: Beta-Binomial rates, a re-run ground truth, and co-failures

**Rates.** Per job, n = runs in which it reached success/failure on its final attempt and
k = final-attempt failures (ADPO's flaky signal) or k = flake events (below). Prior: Jeffreys
Beta(0.5, 0.5); posterior Beta(0.5+k, 0.5+n−k); 95% equal-tailed credible interval. Jobs with
fewer than `N_MIN` runs get no rate and no rank: with Jeffreys, n = 0 gives a posterior mean of
0.5, and small n gives intervals too wide to rank on. `N_MIN` is chosen below from split-half
reliability.

**Ground truth.** A *flake event* is a run (one head_sha) in which a job failed on an earlier
attempt and passed on a later attempt (run_attempt > 1). A job is *re-run-confirmed flaky* if it
has at least one flake event. This truth is incomplete: jobs nobody re-ran can't be confirmed, so
precision against it is a lower bound. It is reported separately for pull-request runs and for
push-to-default-branch runs.

**Co-failure.** For each failing execution: the share of the *other* jobs in the same attempt
that also failed. Failures shared with most of the run point at the commit or the environment
rather than the job.

In [8]:
DEFAULT_BRANCH = {r: meta["repos"][r].get("default_branch") for r in REPOS}
R["default_branch_known"] = sum(v is not None for v in DEFAULT_BRANCH.values())
att = jobs_df[jobs_df.conclusion.isin(["success", "failure"])].copy()
att["run_attempt"] = att.run_attempt.astype(int)
att = att.merge(runs_df[["repo", "run_id", "head_sha", "run_number"]], on=["repo", "run_id"])
att["subset"] = np.where(
    att.event == "pull_request",
    "pull_request",
    np.where((att.event == "push") & (att.branch == att.repo.map(DEFAULT_BRANCH)), "push_default", "other"),
)
assert att.groupby(["repo", "base_run"]).head_sha.nunique().max() == 1  # attempts of a run share one SHA

# Co-failure share of the other jobs in the same attempt.
per_attempt = att.assign(f=att.conclusion == "failure").groupby(["repo", "run_id"]).f.agg(["sum", "size"])
att = att.join(per_attempt, on=["repo", "run_id"])
att["others_failed_share"] = np.where(
    att["size"] > 1, (att["sum"] - (att.conclusion == "failure")) / (att["size"] - 1).clip(lower=1), 0.0
)


def per_run_table(a):
    a = a.sort_values("run_attempt")
    g = a.groupby(["repo", "subset", "name", "base_run"])
    last = g.tail(1).set_index(["repo", "subset", "name", "base_run"])
    first_fail = a[a.conclusion == "failure"].groupby(["repo", "subset", "name", "base_run"]).run_attempt.min()
    last_pass = a[a.conclusion == "success"].groupby(["repo", "subset", "name", "base_run"]).run_attempt.max()
    t = pd.DataFrame({
        "final_failed": last.conclusion == "failure",
        "cofail": last.others_failed_share >= 0.5,
        "attempts": g.run_attempt.nunique(),
    })
    t["flake"] = (last_pass.reindex(t.index) > first_fail.reindex(t.index)).fillna(False).astype(bool)
    return t.reset_index()


prt = per_run_table(att)
R["flake_events_by_subset"] = prt.groupby("subset").flake.sum().astype(int).to_dict()
R["reruns_by_subset"] = (prt.attempts > 1).groupby(prt.subset).sum().astype(int).to_dict()
R["runs_by_subset"] = prt.groupby("subset").base_run.nunique().astype(int).to_dict()


def jeffreys(k, n):
    a, b = 0.5 + k, 0.5 + n - k
    return a / (a + b), stats.beta.ppf(0.025, a, b), stats.beta.ppf(0.975, a, b), stats.beta.ppf(0.05, a, b)


# ---- N_MIN from split-half reliability ----------------------------------------------------
# For each job with >= 2N runs, draw two disjoint random sets of N runs (seeded) and estimate the
# failure rate on each; the Spearman correlation of the two estimates across jobs is the
# reliability of a rate computed from N runs. N_MIN = smallest N reaching 0.7 (a conventional
# "acceptable reliability" level). Averaged over 20 seeded draws.
rel_rng = np.random.default_rng(SEED)
fails_by_job = [g.final_failed.to_numpy() for _, g in prt.groupby(["repo", "name"])]
rel_rows = []
for N in (3, 5, 8, 10, 15, 20, 30, 40, 50):
    pool = [f for f in fails_by_job if len(f) >= 2 * N]
    rhos = []
    for _ in range(20):
        xs, ys = [], []
        for f in pool:
            idx = rel_rng.permutation(len(f))
            xs.append(f[idx[:N]].mean())
            ys.append(f[idx[N : 2 * N]].mean())
        if len(xs) > 5 and np.std(xs) > 0 and np.std(ys) > 0:
            rhos.append(stats.spearmanr(xs, ys).statistic)
    lo, hi = jeffreys(1, N)[1:3]
    rel_rows.append({"N": N, "jobs_with_2N_runs": len(pool),
                     "split_half_spearman": round(float(np.mean(rhos)), 3) if rhos else np.nan,
                     "jeffreys_ci95_width_at_1_failure": round(float(hi - lo), 3)})
rel = pd.DataFrame(rel_rows)
ok = rel[rel.split_half_spearman >= 0.7]
N_MIN = int(ok.N.min()) if len(ok) else int(rel.loc[rel.split_half_spearman.idxmax(), "N"])
R["n_min_table"] = rel.to_markdown(index=False)
R["N_MIN"] = N_MIN
print(rel, "N_MIN =", N_MIN)

    N  jobs_with_2N_runs  split_half_spearman  \
0   3                 14                0.223   
1   5                 14                0.296   
2   8                 14                0.319   
3  10                 13                0.436   
4  15                 13                0.351   
5  20                 13                0.408   
6  30                 12                0.406   
7  40                 12                0.345   
8  50                 12                0.290   

   jeffreys_ci95_width_at_1_failure  
0                             0.785  
1                             0.606  
2                             0.440  
3                             0.370  
4                             0.264  
5                             0.205  
6                             0.142  
7                             0.108  
8                             0.088   N_MIN = 10


### Ranking before/after the posterior (jobs with n >= N_MIN only)

In [9]:
jt = prt.groupby(["repo", "name"]).agg(n=("final_failed", "size"), fails=("final_failed", "sum"),
                                       flakes=("flake", "sum")).reset_index()
R["jobs_below_n_min"] = int((jt.n < N_MIN).sum())
R["jobs_total"] = len(jt)
elig = jt[jt.n >= N_MIN].copy()
elig["raw"] = elig.fails / elig.n
elig["post"], elig["lo"], elig["hi"], elig["lb90"] = jeffreys(elig.fails, elig.n)
fmt = lambda d: d.assign(job=d.name.str[:40], raw_pct=(d.raw * 100).round(1), post_pct=(d.post * 100).round(1),
                         ci95=(d.lo * 100).round(1).astype(str) + "–" + (d.hi * 100).round(1).astype(str))[
    ["repo", "job", "n", "fails", "raw_pct", "post_pct", "ci95"]]
top_raw = elig.sort_values(["raw", "n"], ascending=[False, False]).head(10)
top_post = elig.sort_values("lb90", ascending=False).head(10)
R["fail_top_raw"] = fmt(top_raw).to_markdown(index=False)
R["fail_top_post"] = fmt(top_post).to_markdown(index=False)
R["fail_rank_overlap_top10"] = len(set(map(tuple, top_raw[["repo", "name"]].values))
                                   & set(map(tuple, top_post[["repo", "name"]].values)))

fl_el = elig.copy()
fl_el["raw"] = fl_el.flakes / fl_el.n
fl_el["post"], fl_el["lo"], fl_el["hi"], fl_el["lb90"] = jeffreys(fl_el.flakes, fl_el.n)
R["flake_jobs_with_event"] = int((jt.flakes > 0).sum())
R["flake_top_post"] = (fmt(fl_el[fl_el.flakes > 0].sort_values("lb90", ascending=False).head(10).rename(
    columns={"flakes": "fails"})).rename(columns={"fails": "flake_events"}).to_markdown(index=False))

fig, ax = plt.subplots(figsize=(6.8, 3.8))
show = elig[elig.fails > 0].sort_values("raw", ascending=False).head(25).iloc[::-1]
y = np.arange(len(show))
ax.errorbar(show.post * 100, y, xerr=[(show.post - show.lo) * 100, (show.hi - show.post) * 100],
            fmt="o", ms=3, color="#4C72B0", label="Jeffreys posterior mean, 95% CrI")
ax.scatter(show.raw * 100, y, marker="x", color="crimson", s=14, label="raw rate")
ax.set_yticks(y, [f"{r.split('/')[1]}: {j[:28]} (n={n})" for r, j, n in zip(show.repo, show.name, show.n)], fontsize=6)
ax.set(xlabel="final-attempt failure rate (%)", title=f"Top 25 jobs by raw failure rate (n ≥ {N_MIN})")
ax.legend(frameon=False, fontsize=7)
savefig("e1_failure_posterior")

### ADPO's flaky flags vs the re-run ground truth, by trigger

`current` runs the real `FailureRetryAnalyzer` on each subset of runs. The variants re-apply its
rule to the same final-attempt outcomes (the `current` simulation is asserted equal to the
analyzer's output first): `jeffreys_gate` needs n >= N_MIN and a 90% one-sided Jeffreys lower
bound >= 5%; `drop_cofail` removes runs whose failing final attempt had >= 50% of other jobs also
failing; `both` combines them.

In [10]:
cfgf = FailureRetryConfig()


def rule(g, variant):
    if variant in ("drop_cofail", "both"):
        g = g[~(g.final_failed & g.cofail)]
    n, k = len(g), int(g.final_failed.sum())
    if n == 0 or k == 0:
        return False
    rate = k / n
    if rate >= cfgf.chronic_failure_rate:
        return False
    if variant in ("jeffreys_gate", "both"):
        return n >= N_MIN and stats.beta.ppf(0.05, 0.5 + k, 0.5 + n - k) >= cfgf.flaky_min_rate
    return n >= cfgf.min_sample_size and rate >= cfgf.flaky_min_rate


def subset_runs(repo, subset):
    keep = []
    for r in RUNS[repo]:
        s = "pull_request" if r.event == "pull_request" else (
            "push_default" if r.event == "push" and r.branch == DEFAULT_BRANCH[repo] else "other")
        if s == subset:
            keep.append(r)
    return keep


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


VARIANTS = ["current", "jeffreys_gate", "drop_cofail", "both"]
pr_rows = []
for subset in ("push_default", "pull_request"):
    sp = prt[prt.subset == subset]
    groups = {key: g for key, g in sp.groupby(["repo", "name"])}
    truth = {key for key, g in groups.items() if g.flake.any()}
    rerun_seen = {key for key, g in groups.items() if (g.attempts > 1).any()}
    actual = set()
    for repo in REPOS:
        for f in FailureRetryAnalyzer().analyze(subset_runs(repo, subset)):
            if f.metrics.get("classification") == "flaky":
                actual.add((repo, f.metrics["job_name"]))
    flagged = {v: {key for key, g in groups.items() if rule(g, v)} for v in VARIANTS}
    assert flagged["current"] == actual, (subset, len(flagged["current"] ^ actual))
    for v in VARIANTS:
        fset = flagged[v]
        tp = len(fset & truth)
        p_lo, p_hi = wilson(tp, len(fset))
        r_lo, r_hi = wilson(tp, len(truth))
        obs = fset & rerun_seen
        pr_rows.append({
            "subset": subset, "variant": v, "flagged": len(fset), "confirmed_flaky": len(truth), "tp": tp,
            "precision_lb": round(tp / len(fset), 3) if fset else np.nan,
            "precision_95ci": f"{p_lo:.2f}–{p_hi:.2f}" if fset else "",
            "precision_among_rerun_jobs": round(len(obs & truth) / len(obs), 3) if obs else np.nan,
            "recall": round(tp / len(truth), 3) if truth else np.nan,
            "recall_95ci": f"{r_lo:.2f}–{r_hi:.2f}" if truth else "",
        })
prdf = pd.DataFrame(pr_rows)
R["flaky_pr_table"] = prdf.to_markdown(index=False)
R["flaky_pr"] = prdf.set_index(["subset", "variant"]).to_dict(orient="index")
print(prdf)

         subset        variant  flagged  confirmed_flaky  tp  precision_lb  \
0  push_default        current       11                0   0           0.0   
1  push_default  jeffreys_gate        1                0   0           0.0   
2  push_default    drop_cofail        1                0   0           0.0   
3  push_default           both        1                0   0           0.0   
4  pull_request        current       12                0   0           0.0   
5  pull_request  jeffreys_gate       12                0   0           0.0   
6  pull_request    drop_cofail        3                0   0           0.0   
7  pull_request           both        2                0   0           0.0   

  precision_95ci  precision_among_rerun_jobs  recall recall_95ci  
0      0.00–0.26                         NaN     NaN              
1      0.00–0.79                         NaN     NaN              
2      0.00–0.79                         NaN     NaN              
3      0.00–0.79             

### Co-failure across all repos

In [11]:
cur_flag = set()
for subset in ("push_default", "pull_request"):
    sp = prt[prt.subset == subset]
    cur_flag |= {(subset, *key) for key, g in sp.groupby(["repo", "name"]) if rule(g, "current")}
ff = prt[prt.final_failed].copy()
ff["flagged"] = [(s, r, n) in cur_flag for s, r, n in zip(ff.subset, ff.repo, ff.name)]
fx = ff[ff.flagged]
R["cofail"] = {
    "failing_executions_of_flagged_jobs": len(fx),
    "pct_cofail_ge_half": round(float(fx.cofail.mean() * 100), 1) if len(fx) else None,
    "pct_cofail_by_subset": (fx.groupby("subset").cofail.mean() * 100).round(1).to_dict(),
    "flagged_jobs": len(cur_flag),
    "flagged_jobs_majority_cofail": int((fx.groupby(["subset", "repo", "name"]).cofail.mean() > 0.5).sum()),
    "repos_with_flagged_jobs": int(fx.repo.nunique()),
    "repos_majority_cofail": int((fx.groupby("repo").cofail.mean() > 0.5).sum()),
}
cof_repo = (fx.groupby("repo").cofail.mean() * 100).round(0)
R["cofail_by_repo"] = cof_repo.to_dict()
ev = att.groupby("subset").apply(lambda g: (g.conclusion == "failure").mean() * 100, include_groups=False).round(1)
R["fail_rate_by_subset_pct"] = ev.to_dict()

fig, ax = plt.subplots(figsize=(6, 3.4))
cof_repo.sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.axvline(50, color="k", lw=0.7, ls="--")
ax.set(xlabel="% of flagged jobs' failing runs where ≥ half the other jobs also failed", ylabel="",
       title="Co-failure share, ADPO-flagged 'flaky' jobs")
ax.tick_params(axis="y", labelsize=7)
savefig("e2_cofailure_by_repo")
print(R["cofail"], R["fail_rate_by_subset_pct"])

{'failing_executions_of_flagged_jobs': 328, 'pct_cofail_ge_half': 65.5, 'pct_cofail_by_subset': {'pull_request': 65.3, 'push_default': 67.7}, 'flagged_jobs': 23, 'flagged_jobs_majority_cofail': 18, 'repos_with_flagged_jobs': 1, 'repos_majority_cofail': 1} {'other': 22.7, 'pull_request': 21.2, 'push_default': 10.8}


## f. Precision check (hand labels)

All findings at default settings, sampled with a fixed seed, stratified so each analyzer that
produced findings is represented (40 total, split as evenly as availability allows). Labels are
filled in by hand in `labels.csv` (useful / not useful / wrong); the next cell computes precision.

In [12]:
engine = AnalysisEngine()
all_f = []
for repo in REPOS:
    for f in engine.run(RUNS[repo]):
        all_f.append({"repo": repo, "analyzer": f.analyzer_type, "title": f.title, "f": f})
    assert not engine.errors, engine.errors
all_f = sorted(all_f, key=lambda d: (d["repo"], d["analyzer"], d["title"]))
for i, d in enumerate(all_f):
    d["finding_id"] = f"F{i:04d}"
fdf = pd.DataFrame(all_f)
R["findings_default"] = fdf.groupby("analyzer").size().to_dict()
R["findings_default_total"] = len(fdf)
R["findings_per_repo"] = fdf.groupby("repo").size().reindex(REPOS, fill_value=0).to_dict()

rng = np.random.default_rng(SEED)
avail = fdf.groupby("analyzer").size().to_dict()
alloc = {a: 0 for a in avail}
remaining = 40
while remaining > 0 and any(alloc[a] < avail[a] for a in avail):
    for a in sorted(avail):
        if remaining and alloc[a] < avail[a]:
            alloc[a] += 1
            remaining -= 1
sample = pd.concat([fdf[fdf.analyzer == a].iloc[rng.choice(avail[a], alloc[a], replace=False)] for a in sorted(alloc)])
R["label_alloc"] = alloc

labels_path = HERE / "labels.csv"
existing = pd.read_csv(labels_path, dtype=str).fillna("") if labels_path.exists() else None
has_labels = existing is not None and (existing.label.str.strip() != "").any()
if has_labels and set(existing.finding_id) != set(sample.finding_id):
    raise RuntimeError("labels.csv holds hand labels for a different sample (different data source?); refusing to overwrite it")
if existing is None or set(existing.finding_id) != set(sample.finding_id):
    pd.DataFrame({
        "finding_id": sample.finding_id,
        "repo": sample.repo,
        "analyzer": sample.analyzer,
        "evidence_summary": [d.title + " | " + " | ".join(d.evidence[:3]) for d in sample.f],
        "label": "",
        "notes": "",
    }).to_csv(labels_path, index=False)
(HERE / "data" / "sampled_findings.json").write_text(json.dumps(
    [{"finding_id": i, "repo": r, **f.to_dict()} for i, r, f in zip(sample.finding_id, sample.repo, sample.f)],
    indent=1, default=str) + "\n")
print(alloc, len(sample))

{'failure_retry': 13} 13


In [13]:
lab = pd.read_csv(labels_path, dtype=str).fillna("")
lab["label"] = lab.label.str.strip().str.lower()
done = lab[lab.label != ""]
if done.empty:
    R["precision_table"] = "_Not computed yet: `labels.csv` has no labels. Fill in the `label` column and re-run._"
else:
    rows = []
    for a, g in list(done.groupby("analyzer")) + [("ALL", done)]:
        k, n = int((g.label == "useful").sum()), len(g)
        lo, hi = wilson(k, n)
        rows.append({"analyzer": a, "labelled": n, "useful": k, "precision": round(k / n, 2),
                     "wilson_95_lo": round(lo, 2), "wilson_95_hi": round(hi, 2)})
    R["precision_table"] = pd.DataFrame(rows).to_markdown(index=False)
print(R["precision_table"])

_Not computed yet: `labels.csv` has no labels. Fill in the `label` column and re-run._
